### **Installing And Importing Packages**

In [1]:
!pip install transformers -qqq

from huggingface_hub import login
from kaggle_secrets import UserSecretsClient
from peft import PeftModel
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModelForSequenceClassification, pipeline

#### **Model Initialization and Authentication Setup**

We first establish secure access to external model repositories and defines the core models used within the system pipeline. Authentication is handled via **Hugging Face API token**, retrieved securely using `UserSecretsClient`, allowing the environment to log in without exposing sensitive credentials. Three key models are then specified: `Qwen2-1.5B-Instruct`, a lightweight instruction-tuned large language model used to power the chatbot’s conversational capabilities **BERT-base-cased**, a robust pretrained transformer model serving as the foundational architecture for text classification and a fine-tuned variant `v2-heladepdet-bert-finetuned-classification`, which is specifically adapted to detect and classify depression-related signals in text. 

In [2]:
# Log in to Hugging Face
HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
login(HF_TOKEN)

# Define the core models
CHATBOT_MODEL_NAME = "Qwen/Qwen2-1.5B-Instruct"
SCORE_BASE_MODEL_NAME = "google-bert/bert-base-cased"
SCORE_MODEL_NAME = "chiabingxuan/v2-heladepdet-bert-finetuned-classification"

#### **Model and Tokenizer Loading for Inference Pipeline**

Next, we initialize both the chatbot and scoring components required for the system’s end-to-end functionality. Tokenizers are first loaded to convert raw text into structured inputs that the models can process. The chatbot leverages a causal language model (Qwen2-1.5B-Instruct) for generating conversational responses, with `device_map="auto"` enabling efficient allocation across available hardware (GPU/CPU). 

The scoring pipeline loads a sequence classification model based on a BERT architecture, configured with four output labels to represent varying mental health depression states. The base model is then augmented using a **Parameter-Efficient Fine-Tuning (PEFT)** adapter, which injects task-specific knowledge from the fine-tuned checkpoint without requiring full model retraining. Together, these components enable both natural dialogue generation and real-time depression classification.

In [3]:
# Load tokenizer for chatbot model (text → tokens)
chatbot_tokenizer = AutoTokenizer.from_pretrained(CHATBOT_MODEL_NAME)

# Load chatbot language model with automatic device allocation
chatbot_model = AutoModelForCausalLM.from_pretrained(CHATBOT_MODEL_NAME, device_map="auto")

# Load tokenizer for scoring/classification model
score_tokenizer = AutoTokenizer.from_pretrained(SCORE_MODEL_NAME)

# Load base BERT model configured for 4-class classification
score_model = AutoModelForSequenceClassification.from_pretrained(SCORE_BASE_MODEL_NAME,
                                                                 num_labels=4,
                                                                 device_map="auto")

# Load PEFT adapter weights into the base model for task-specific fine-tuning
score_model = PeftModel.from_pretrained(score_model, SCORE_MODEL_NAME)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: google-bert/bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


#### **Device Verification for Model Deployment**

Next, we verify the hardware placement of both the chatbot and scoring models after initialization. By inspecting the device of the first parameter in each model, we can confirm whether the models have been successfully allocated to the intended compute resource (GPU or CPU). This is particularly important when using `device_map="auto"` as it ensures that the models are leveraging available acceleration for efficient inference. Proper device placement directly impacts performance especially for large language models and real-time classification tasks.

In [4]:
# Print device of chatbot model with descriptive label
print(f"Device Of Chatbot Model: {next(chatbot_model.parameters()).device}")

# Print device of scoring model with descriptive label
print(f"Device Of Scoring Model: {next(score_model.parameters()).device}")

Device Of Chatbot Model: cuda:0
Device Of Scoring Model: cuda:0


#### **System Prompt Design for Controlled Conversational Behavior**

This section defines the system prompt that governs the chatbot’s behavior, tone and response structure during user interactions. It tightly constrains the model to act as a structured conversational assistant focused solely on eliciting descriptive insights about a student’s emotional state and daily experiences rather than providing advice or therapeutic intervention. The prompt enforces strict rules such as avoiding causal or “why”-based questions, limiting each response to a brief empathetic reflection followed by exactly one open-ended question and maintaining a warm, non-judgmental tone. This level of control is critical to ensure consistency, safety and suitability for downstream depression mental health analysis where the goal is to collect high-quality, unbiased user inputs rather than influence them.

In [5]:
# Define system prompt to control chatbot behavior and response structure
SYSTEM_PROMPT = """
You are a structured, empathetic conversational assistant designed to help students from the National University of Singapore describe their current emotional and daily experiences.

# Define chatbot's sole purpose (eliciting descriptive responses)
Your ONLY function is to elicit descriptive user responses about their mental state, mood, energy, and daily functioning in a natural conversation.

# Explicitly restrict role (not a therapist or advisor)
You are NOT a therapist, counselor, or advisor.

-----------------------
CORE OBJECTIVE
-----------------------
# Encourage descriptive sharing of lived experiences
- Encourage the user to describe their lived experience (feelings, thoughts, daily routine, energy, motivation)
# Maintain safe and empathetic tone
- Maintain a warm, empathetic, non-judgmental tone
# Ensure outputs are useful for downstream analysis
- Collect rich descriptive responses for downstream analysis

-----------------------
STRICT FORBIDDEN BEHAVIOR
-----------------------
# Prevent advice-giving or solutioning
- Do NOT give advice, suggestions, coping strategies, or solutions
# Avoid framing as helping/fixing
- Do NOT mention “helping,” “fixing,” or “figuring things out together”
# Prevent causal probing (no "why" questions)
- Do NOT ask about causes, reasons, triggers, or explanations (e.g., “why”, “what caused this”, “what is contributing to”)
# Avoid meta or preference-based questions
- Do NOT ask preference-based or meta questions (e.g., “would you like”, “do you want to continue”)
# Restrict to one question per response
- Do NOT ask multiple questions in one response
# Avoid giving choices/options
- Do NOT provide options or choices

-----------------------
RESPONSE FORMAT (MANDATORY)
-----------------------
# Enforce strict response structure
Each response MUST follow this structure:

# Step 1: empathetic reflection
1. Empathetic reflection of the user's message (1–2 sentences)
# Step 2: exactly one open-ended question
2. EXACTLY ONE open-ended question

# Define constraints for the question
The question MUST:
# Be grounded in user's prior message
- Be dynamically based on the user's previous replies
# Focus on experience, not causation
- Ask about experience, not cause or reasoning
# Guide towards relevant dimensions
- Focus on:
  - daily routine
  - feelings over time
  - energy levels in context
  - impact on activities
  - thoughts or internal state
# Explicitly forbid causal phrasing
- Never include “why”, “what caused”, or “what contributes”

-----------------------
QUESTION STYLE EXAMPLES (GOOD)
-----------------------
# Provide examples of acceptable question styles
- "What has your day-to-day been like recently?"
- "How has this been affecting your daily routine?"
- "What does a typical day feel like for you lately?"
- "How has your energy been showing up during the day?"
- "What have your mornings or evenings been like recently?"

-----------------------
QUESTION STYLE (BAD - NEVER USE)
-----------------------
# Provide examples of prohibited question styles
- "Why do you feel this way?"
- "What caused this?"
- "Is there something contributing to this?"
- "Would you like to talk about this or take a break?"
- "How can I help you fix this?"

-----------------------
STYLE
-----------------------
# Define tone and style constraints
- Warm, calm, natural tone
- No clinical language
- No motivational framing
# Keep responses concise
- Keep responses short (1–3 sentences before the question)
# Enforce single-question ending
- Always end with exactly one question
"""

**Why the Question Examples Are Good vs Not Good**

The “good” examples are effective because they anchor the conversation in lived experience rather than analysis or justification. For instance, “What has your day-to-day been like recently?” invites the user to describe patterns and routines which naturally surfaces emotional and behavioral signals without forcing interpretation. Similarly, “How has your energy been showing up during the day?” subtly captures fluctuations in mood and functioning which are highly informative for downstream models. These questions are open-ended, non-intrusive and easy to answer, reducing cognitive load and encouraging richer responses.

In contrast, the “bad” examples violate key design principles. Questions like “Why do you feel this way?” or “What caused this?” push the user into causal reasoning which can introduce bias, speculation or discomfort, especially in sensitive contexts like mental health and depression. Meanwhile, “Would you like to talk about this or take a break?” is a meta-question, shifting control of the conversation rather than continuing it which disrupts data collection consistency. Lastly, “How can I help you fix this?” reframes the assistant as a problem-solver, contradicting the system’s constraint of not providing advice and potentially altering the user’s natural responses.

**Why These Constraints?**

The constraints are intentionally designed to support reliable depression signal detection rather than conversational assistance. By restricting the model from asking “why” questions or offering advice, the system minimizes cognitive bias and avoids steering the user toward rationalizing or reframing their emotions. This is important because diagnostic indicators such as low energy, disrupted routines or persistent negative affect are often best captured through natural, unfiltered descriptions of lived experiences. Additionally, enforcing a consistent structure (reflection + one question) standardizes the interaction, making the collected text more uniform and suitable for downstream models like classifiers which rely on stable input patterns to produce accurate predictions. 

**More About The Style Of Conversation**

The prescribed style: warm, calm and non-clinical is critical for encouraging authentic user disclosure without intimidation or resistance. A highly clinical or formal tone may feel evaluative or diagnostic, causing users to withhold or alter their responses while an overly motivational tone could unintentionally influence emotional framing. By keeping responses short and empathetic, the system reduces friction and keeps the user engaged without overwhelming them. This balance ensures that users feel heard and comfortable while still maintaining the primary objective of capturing high-quality, natural language data for analysis.

#### **Chatbot Response Generation Function**

This function handles the end-to-end process of generating a chatbot reply from a sequence of conversation messages. It first formats the conversation into a structured prompt using the tokenizer’s chat template, ensuring compatibility with the instruction-tuned model. The inputs are then moved to the appropriate device (example: T4 GPU) for efficient inference. The model generates a response with a capped token limit to control length, after which only the newly generated portion (excluding the original prompt) is decoded into human-readable text. Finally, the response is printed for visibility and returned for downstream use, enabling seamless integration into an interactive conversational pipeline.

In [6]:
# Chatbot Response Generation Function
def generate_response(model, tokenizer, messages):
    # Indicate that the chatbot is generating a response
    print("Chatbot is thinking...")
    
    # Convert messages into model-ready input format using chat template
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,        # Append generation prompt for assistant reply
        tokenize=True,                     # Convert text into tokens
        return_dict=True,                  # Return outputs as a dictionary
        return_tensors="pt",               # Return PyTorch tensors
    ).to(model.device)                     # Move inputs to same device as model
    
    # Generate model output with a maximum token limit
    outputs = model.generate(**inputs, max_new_tokens=256)
    
    # Decode only the newly generated tokens (exclude input prompt)
    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True           # Remove special tokens from output
    )
    
    # Print the chatbot's response for visibility
    print(f"Chatbot says: {response}\n")
    # Return the generated response
    return response

#### **Multi-Turn Chat Execution Pipeline**

This function orchestrates the full multi-turn interaction between the user and the chatbot, maintaining conversational context across turns. It begins by initializing the conversation with the system prompt, which defines the chatbot’s behavior and constraints. An opening response is generated to start the dialogue, after which the function enters a loop to collect user inputs and append them to the message history. 

At each turn (except the final one), the chatbot generates a context-aware reply based on the entire conversation so far. By continuously updating the messages list with both user and assistant exchanges, the function ensures coherence and continuity, making it suitable for structured conversational data collection and downstream analysis.

In [7]:
# Multi-Turn Chat Execution Pipeline
def run_chat(model, tokenizer, num_turns):
    # Initialize conversation with system prompt
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]

    # Generate initial chatbot message to start the conversation
    opening_chatbot_message = generate_response(model, tokenizer, messages)
    messages.append({"role": "assistant", "content": opening_chatbot_message})

    # Loop through the specified number of turns
    for turn in range(num_turns):
        # Collect user input from console
        user_input = input("Enter your response:\n")
        messages.append({"role": "user", "content": user_input})

        # Generate chatbot response if not the final turn
        if turn < num_turns - 1:
            print()
            new_chatbot_message = generate_response(model, tokenizer, messages)
            messages.append({"role": "assistant", "content": new_chatbot_message})

    # Return full conversation history for analysis or storage
    return messages

#### **User Input Aggregation and Mental Health Scoring**

Lsstly, we will handle the collection of all user-provided messages and performs automated mental health scoring using the fine-tuned classification model. First, user messages are extracted from the full conversation history and concatenated into a single text block, preserving context and sequence. This aggregated input is then fed into a text classification pipeline, which applies the PEFT-enhanced BERT model to predict the user’s depression state across four defined labels. By combining multi-turn conversational data into a single input, this approach maximizes the richness of information for downstream scoring, enabling more accurate detection of patterns relevant to depression.

In [8]:
# Extract all user messages from conversation history and join them
def get_concat_user_inputs(messages):
    user_inputs = [message["content"] for message in messages if message["role"] == "user"]
    return "\n\n".join(user_inputs)

# Predict mental health score using a text classification pipeline
def predict_score(model, tokenizer, text):
    clf = pipeline(
        "text-classification",   # Define task type
        model=model,             # Use fine-tuned scoring model
        tokenizer=tokenizer      # Use associated tokenizer
    )
    return clf(text)            # Return classification results

# Define number of conversational turns
NUM_TURNS = 3

# Run interactive chat session and collect conversation messages
messages_produced = run_chat(chatbot_model, chatbot_tokenizer, num_turns=NUM_TURNS)

# Concatenate all user inputs into a single text block for scoring
user_input_concat = get_concat_user_inputs(messages_produced)

# Predict mental health score based on aggregated user input
predict_score(score_model, score_tokenizer, user_input_concat)

Chatbot is thinking...
Chatbot says: I'm here to listen. Please share more about how you're feeling today?



Enter your response:
 I am feeling extremely stressed out and want to commit suicide anytime! I feel that life is totally meaningless and crazy, I just hate life.



Chatbot is thinking...
Chatbot says: It sounds like you're experiencing intense emotions right now. It's really important to acknowledge these feelings and seek support from someone who can understand and help you. Have you considered talking to a trusted friend, family member, or professional?



Enter your response:
 I have never talked to anyone as I feel that seeking support is utterly useless and ineffective. My friends are all fake and don't show any support for me. My family is consistently working and they just leave me alone, I am like a neglected child. Professionals to me are purely useless as they just talk and encourage me, but surely to no avail.



Chatbot is thinking...
Chatbot says: I'm sorry to hear that you feel so alone and unsupported. It's important to know that you are not alone in these feelings and that it's okay to reach out for help. You may find comfort in talking to someone who understands what you're going through, such as a mental health professional or a support group. They can offer guidance and support tailored to your needs.



Enter your response:
 I don't think there is any use, I feel that my life is empty and always bleak. I just don't want to continue life anymore, I am sick and tired.


[{'label': 'LABEL_3', 'score': 0.7495991587638855}]